# Context Engineering – Flipkart Customer Support AI

## Objective

Build a context-aware customer support AI system for an e-commerce platform.

This notebook uses exactly three context types:

1. Static Context
2. Dynamic Context
3. External Context

The three contexts are structured separately, assembled intentionally, and passed to the AI model in the final step.

### Key Idea

**Context is infrastructure, not just text.**


## Problem Statement

A customer support AI should not answer a customer using only the customer's message.

For a reliable answer, the system may need:

- Stable company rules and assistant behavior
- Current customer and order information
- Verified information from external services

Example customer request:

> I want to cancel my wireless headphones order.

The AI must use the available order information and cancellation eligibility instead of guessing.

The goal is to provide the right context to the model instead of dumping unrelated information into one prompt.


## Context Architecture

### Static Context

Stable information that defines how the assistant should behave.

Examples:

- Assistant role
- Response rules
- Business rules
- Safety constraints

### Dynamic Context

Information that can change for each customer or request.

Examples:

- Customer ID
- Current orders
- Order status
- Payment status
- Refund status

### External Context

Verified information obtained from systems outside the customer profile.

Examples:

- Logistics service status
- Delivery tracking information
- Payment service status

The three contexts are assembled before the model is called.


## Step 1: Static Context

Static context contains stable instructions and business rules.

It should not contain information that belongs to one particular customer or one particular order.


In [ ]:
STATIC_CONTEXT = {
    "role": "You are a customer support assistant for Flipkart.",
    "rules": [
        "Be polite, professional, and concise.",
        "Do not invent customer, order, payment, refund, or delivery information.",
        "Use the supplied verified context for customer-specific answers.",
        "Follow the supplied business rules.",
        "If required information is missing or conflicting, ask a clarification question.",
        "Do not claim that an action was completed unless the relevant system confirms it."
    ],
    "business_rules": {
        "cancellation": (
            "An order can be cancelled only when its cancellation_eligible "
            "status is true."
        ),
        "refund": (
            "Refund handling depends on the applicable order and refund policy. "
            "Do not promise a refund without sufficient verified information."
        )
    },
    "response_style": "Provide a clear answer and the appropriate next step."
}

print(STATIC_CONTEXT)


{'role': 'You are a customer support assistant for Flipkart.', 'rules': ['Be polite, professional, and concise.', 'Do not invent customer, order, payment, refund, or delivery information.', 'Use the supplied verified context for customer-specific answers.', 'Follow the supplied business rules.', 'If required information is missing or conflicting, ask a clarification question.', 'Do not claim that an action was completed unless the relevant system confirms it.'], 'business_rules': {'cancellation': 'An order can be cancelled only when its cancellation_eligible status is true.', 'refund': 'Refund handling depends on the applicable order and refund policy. Do not promise a refund without sufficient verified information.'}, 'response_style': 'Provide a clear answer and the appropriate next step.'}


## Step 2: Dynamic Context

Dynamic context represents the current customer and transaction state.

This information can change over time and should be refreshed for the current request.


In [ ]:
DYNAMIC_CONTEXT = {
    "customer_id": "C1024",
    "active_orders": [
        {
            "order_id": "FK12345",
            "product": "Running Shoes",
            "status": "Out for delivery",
            "expected_delivery": "Today",
            "cancellation_eligible": False
        },
        {
            "order_id": "FK67890",
            "product": "Wireless Headphones",
            "status": "Processing",
            "expected_delivery": "Tomorrow",
            "cancellation_eligible": True
        }
    ],
    "payment_status": "Payment successful",
    "refund_status": "No active refund"
}

print(DYNAMIC_CONTEXT)


{'customer_id': 'C1024', 'active_orders': [{'order_id': 'FK12345', 'product': 'Running Shoes', 'status': 'Out for delivery', 'expected_delivery': 'Today', 'cancellation_eligible': False}, {'order_id': 'FK67890', 'product': 'Wireless Headphones', 'status': 'Processing', 'expected_delivery': 'Tomorrow', 'cancellation_eligible': True}], 'payment_status': 'Payment successful', 'refund_status': 'No active refund'}


## Step 3: External Context

External context represents verified information obtained from external services.

In a production system, these values would normally come from APIs or other trusted integrations.

For this demonstration, the external service responses are represented as structured Python data.


In [ ]:
EXTERNAL_CONTEXT = {
    "logistics_service": {
        "delivery_partner": "Demo Express",
        "network_status": "Normal",
        "last_scan_location": "Bhopal Hub"
    },
    "payment_service": {
        "service_status": "Operational"
    }
}

print(EXTERNAL_CONTEXT)


{'logistics_service': {'delivery_partner': 'Demo Express', 'network_status': 'Normal', 'last_scan_location': 'Bhopal Hub'}, 'payment_service': {'service_status': 'Operational'}}


## Step 4: Customer Request

The customer request is dynamic because it changes for every interaction.


In [ ]:
USER_QUERY = "I want to cancel my wireless headphones order."

print(USER_QUERY)


I want to cancel my wireless headphones order.


## Step 5: Context Assembly

The context builder combines the three context types with the current customer request.

The model receives a structured representation of the situation rather than an unorganized context dump.


In [ ]:
def build_final_prompt(
    user_query,
    static_context,
    dynamic_context,
    external_context
):
    rules = "\n".join(
        f"- {rule}" for rule in static_context["rules"]
    )

    prompt = f"""
{static_context["role"]}

Assistant Rules:
{rules}

Business Rules:
- Cancellation: {static_context["business_rules"]["cancellation"]}
- Refund: {static_context["business_rules"]["refund"]}

Response Style:
{static_context["response_style"]}

Current Customer and Order Data:
{dynamic_context}

Verified External Service Information:
{external_context}

Customer Request:
{user_query}

Task:
Answer the customer's request using the supplied context.
Do not invent information.
Do not claim that an action has been completed unless the supplied context confirms it.
If the request cannot be resolved from the available information, ask a specific clarification question or explain the required next step.
""".strip()

    return prompt


FINAL_PROMPT = build_final_prompt(
    USER_QUERY,
    STATIC_CONTEXT,
    DYNAMIC_CONTEXT,
    EXTERNAL_CONTEXT
)

print(FINAL_PROMPT)


You are a customer support assistant for Flipkart.

Assistant Rules:
- Be polite, professional, and concise.
- Do not invent customer, order, payment, refund, or delivery information.
- Use the supplied verified context for customer-specific answers.
- Follow the supplied business rules.
- If required information is missing or conflicting, ask a clarification question.
- Do not claim that an action was completed unless the relevant system confirms it.

Business Rules:
- Cancellation: An order can be cancelled only when its cancellation_eligible status is true.
- Refund: Refund handling depends on the applicable order and refund policy. Do not promise a refund without sufficient verified information.

Response Style:
Provide a clear answer and the appropriate next step.

Current Customer and Order Data:
{'customer_id': 'C1024', 'active_orders': [{'order_id': 'FK12345', 'product': 'Running Shoes', 'status': 'Out for delivery', 'expected_delivery': 'Today', 'cancellation_eligible': False}

## Step 6: Validate the Relevant Order

Before the model call, the application can check whether the customer request maps to one unique order.

This prevents the application from silently selecting an order when multiple matching orders exist.


In [ ]:
def find_order_by_product(product_name, orders):
    product_name = product_name.lower()

    matches = [
        order
        for order in orders
        if product_name in order["product"].lower()
    ]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        return None

    return "AMBIGUOUS"


relevant_order = find_order_by_product(
    "headphones",
    DYNAMIC_CONTEXT["active_orders"]
)

if relevant_order == "AMBIGUOUS":
    print("Clarification required: multiple matching orders were found.")
elif relevant_order is None:
    print("Clarification required: no matching order was found.")
else:
    print("Relevant order:", relevant_order)


Relevant order: {'order_id': 'FK67890', 'product': 'Wireless Headphones', 'status': 'Processing', 'expected_delivery': 'Tomorrow', 'cancellation_eligible': True}


## Step 7: API Configuration

The model call follows the OpenAI-compatible approach used in the reference lab.

The API key is read from Google Colab Secrets instead of being written directly into the notebook.

Store the key in Colab Secrets with the name:

`api_key`


In [ ]:
import os

# Load the API key securely from Google Colab Secrets.
try:
    from google.colab import userdata
    MY_API_KEY = userdata.get("api_key")
except ImportError:
    # For local/Jupyter use, set API_KEY as an environment variable.
    MY_API_KEY = os.getenv("API_KEY")

if not MY_API_KEY:
    raise ValueError(
        "API key not found. Add `api_key` to Colab Secrets or set the `API_KEY` environment variable."
    )

print("API key loaded successfully.")


API key loaded successfully.


## Final Call

This is the final step of the workflow.

The model receives:

- Static Context
- Dynamic Context
- External Context
- Current Customer Request

No Memory Context is used in this implementation.


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=MY_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": STATIC_CONTEXT["role"]
        },
        {
            "role": "user",
            "content": FINAL_PROMPT
        }
    ]
)

print(response.choices[0].message.content)


Your order for Wireless Headphones (Order ID: FK67890) is eligible for cancellation.

Would you like me to proceed with cancelling this order for you?
